# LeRobot Workshop — From Demonstration to Robot Autonomy

Today you'll walk the whole path a real robot-learning project takes:

**Record demonstrations → build a dataset → train a policy → run it on the robot**

We won't have GPUs at every seat, so the plan is:

1. See how a dataset gets recorded (with LeLab) — conceptually, so you understand where the data comes from.
2. Explore a dataset we already recorded — a real one, public on the Hugging Face Hub.
3. See, in broad strokes, how that dataset became a trained policy (SmolVLA) — again, we already did the actual training.
4. Connect to a shared GPU and run that trained policy on your own robot arm, live.

Then: a challenge, in a separate notebook, where you build something on top of it yourselves.

## Part 1 — Recording your own dataset (with LeLab)

This is the part of the pipeline that happens *before* any of today's shared resources exist. If you had your own robot arm and camera at home, here's the shape of it — **what** to do, not the exact clicks (LeLab's own guided flow will walk you through each step):

- **Install LeLab** on the machine connected to your robot.
- **Connect both arms** (leader and follower — the one that moves) and the camera.
- **Calibrate** each arm inside LeLab. This is a short guided step per arm; LeLab tells you exactly what to move and when.
- **Start a recording session**: give it a task description in plain English (e.g. *"pick up the letter M and place it in the first box"*), how many episodes to record, and how long each one should run.
- **Perform the task** with the leader arm, the same way, several times. Consistency matters far more than speed — the policy will learn *your* pattern.
- LeLab saves the episodes into a dataset for you, and can push it straight to the Hugging Face Hub.

That's genuinely it — no code required for this part. We're not recording live today so everyone gets to the same starting point; instead, in Part 2 we'll open a dataset recorded exactly this way.

## Part 2 — Explore a real LeRobot dataset

We recorded 30 episodes of this exact task — pick up a letter (M, H, or P) and place it in its box — and merged them into one public dataset:

**`sohrabark/abc_mhp_v2_merged_20260805`**

### Open it in the Hugging Face dataset viewer

👉 https://huggingface.co/datasets/sohrabark/abc_mhp_v2_merged_20260805

Every Hugging Face dataset gets an automatic **Dataset Viewer** tab — click through a few rows.

### Open it in the LeRobot visualizer

LeRobot ships a purpose-built visualizer as a Hugging Face Space:

👉 https://huggingface.co/spaces/lerobot/visualize_dataset

Paste in `sohrabark/abc_mhp_v2_merged_20260805` where it asks for a dataset repo id. This is the more useful view for robotics data — you get:
- the camera video, playing back per episode
- the **action** and **observation.state** curves (one line per joint) scrubbing in sync with the video
- an episode picker, so you can compare episode 0 against episode 29

### Things worth noticing while you look around

- The dataset has **3 different task descriptions** — episodes 0-9 say "pick up M", 10-19 say "pick up H", 20-29 say "pick up P". One dataset, three tasks, because the arm always starts and ends near the same rest pose regardless of which letter it's doing.
- Watch a joint curve (e.g. `shoulder_lift`) over one episode: it's *flat* for the first second or two (the idle moment before the demonstrator starts moving), then sweeps out to pick up the letter, then returns close to where it started.
- Every episode's camera feed looks almost identical at frame 0 — same framing, same lighting, same starting pose. That consistency is exactly what part 1 was asking you to nail.

In [1]:
# Optional: peek at the dataset's basic stats without downloading it,
# using only the Hub API (works even without the `lerobot` package installed).

from huggingface_hub import HfApi

api = HfApi()
info = api.dataset_info("sohrabark/abc_mhp_v2_merged_20260805")
print("private:", info.private)
print("files:", len(info.siblings))


/Users/akeshavarzi/dev/mhp/PAI-2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


private: False
files: 8


## Part 3 — How this dataset became a trained policy

We fine-tuned **SmolVLA** — a small (~500M parameter) vision-language-action model that LeRobot publishes a pretrained base checkpoint for (`lerobot/smolvla_base`) — on the dataset from Part 2.

The idea in one sentence: SmolVLA already knows *roughly* how to look at a camera image and a language instruction and produce robot joint targets, from having been pretrained on many robots and tasks; fine-tuning specializes it to *our* dataset, *our* camera, *our* arm.

The actual command we ran (informational — you won't run this today, there's no GPU at your seat):

```bash
lerobot-train \
  --dataset.repo_id=sohrabark/abc_mhp_v2_merged_20260805 \
  --policy.path=lerobot/smolvla_base \
  --policy.device=cuda \
  --steps=20000 \
  --batch_size=64 \
  --save_freq=5000 \
  --rename_map='{"observation.images.GrispCamera":"observation.images.camera1"}' \
  --wandb.enable=false \
  --policy.push_to_hub=true \
  --policy.repo_id=sohrabark/smolvla_abc_mhp_v2_merged_20260805
```

What each piece roughly means:
- `--dataset.repo_id` — the dataset from Part 2.
- `--policy.path=lerobot/smolvla_base` — start from the pretrained checkpoint, don't train from scratch.
- `--steps` / `--batch_size` — how many gradient updates, and how many examples per update. 20,000 steps × batch 64 over ~25k frames works out to roughly 51 passes over the dataset.
- `--rename_map` — our camera is called `GrispCamera`; SmolVLA expects a generically-named `camera1`, so we rename it at training time.
- `--policy.push_to_hub` — when training finishes, upload the result to the Hub automatically.

What actually happened when we ran it: **~96 minutes** on a single cloud GPU (an NVIDIA L4, 24GB), training loss dropped smoothly from ~0.7 down to **~0.031**.

The result is public on the Hub:

**`sohrabark/smolvla_abc_mhp_v2_merged_20260805`**

You'll use that exact checkpoint in the next part — running on a GPU we've already set up, not yours.

## Part 4 — Run the trained policy on your robot

Here's the piece that makes this workable without a GPU at every seat: the trained policy runs on **one shared cloud GPU**, and your laptop runs a *thin client* — it streams your camera image and your arm's current joint positions over the network, and gets back where to move next, live, at 30 times per second.

```
your laptop                                    shared GPU (cloud)
┌─────────────────────────┐                    ┌───────────────────────┐
│ SO-101 arm + camera     │   SSH tunnel       │ SmolVLA policy server │
│ robot_client            ┼───────────────────►│ (smolvla_abc_mhp_v2_merged_20260805)
│                         ┼◄───────────────────┤                       │
└─────────────────────────┘   actions @ 30Hz   └───────────────────────┘
```

### Two robot arms, one GPU

We have two LeRobot arm sets and one GPU box for the whole room. One GPU can run the policy for both arms at once — we just gave each table **its own port** on the same machine, so the two tables can't interfere with each other:

| Table | Port |
|---|---|
| Table A | `8080` |
| Table B | `8081` |

Ask an organizer which table you're on if you're not sure.

### Connect

**Step 1 — open a tunnel to the shared GPU:**

```bash
ssh -N -L <PORT>:localhost:<PORT> ubuntu@<SERVER_IP>
```
Leave this running in its own terminal for the whole workshop — it's your private pipe into the shared GPU. `<PORT>` is whichever port your table was assigned (see above).

`<SERVER_IP>` will be given out at the start of the workshop.

**Step 2 — run one task:**

```bash
python -m lerobot.async_inference.robot_client \
  --robot.type=so101_follower \
  --robot.port=<YOUR_ROBOT_PORT> \
  --robot.id=<YOUR_CALIBRATION_ID> \
  --robot.cameras="{ camera1: {type: opencv, index_or_path: <YOUR_CAMERA_INDEX>, width: 640, height: 480, fps: 30}}" \
  --server_address=127.0.0.1:<PORT> \
  --policy_type=smolvla \
  --pretrained_name_or_path=sohrabark/smolvla_abc_mhp_v2_merged_20260805 \
  --policy_device=cuda \
  --client_device=cpu \
  --actions_per_chunk=50 \
  --chunk_size_threshold=0.5 \
  --aggregate_fn_name=weighted_average \
  --fps=30 \
  --task="Pick up the M letter and place it in the first box."
```

Notes:
- `--robot.port`, `--robot.id`, `--robot.cameras` are the same values LeLab used for your calibration in Part 1.
- `--server_address` points at your **local** end of the SSH tunnel — `127.0.0.1`, not `<SERVER_IP>` — the tunnel does the forwarding.
- This command **runs until you press Ctrl-C** — it has no built-in stopping point. Let it run for ~20-25 seconds (that's roughly how long the training recordings were), then stop it.
- Swap `--task` for any of the three exact strings from the table above to try a different letter.

### The three tasks this policy knows

| Letter | Task string (exact) | Goes into |
|---|---|---|
| **M** | `Pick up the M letter and place it in the first box.` | 1st box |
| **H** | `Pick up the H letter and place it in the second box.` | 2nd box |
| **P** | `Pick up the P letter and place it in the third box.` | 3rd box |

Only these three letters were trained. Anything else is not something the policy has ever seen.

Try running one. Watch what the arm actually does versus what you expected — that observation is useful for the challenge ahead.

## What's next

Open **`workshop_2_challenge.ipynb`**. You have everything you need already: the tunnel command and the single-task inference command above. The challenge is what you build on top of them.